In [ ]:
# === Tutorial bootstrap: fetch utils/ + sample data if missing (for Colab blob links) ===
import os, sys, urllib.request

REPO   = "amirfar76/neurips25-valid-hparam-selection"
BRANCH = "main"
BASE   = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}"

def ensure_utils():
    os.makedirs("utils", exist_ok=True)
    for fname in ["csvio.py", "testing.py"]:
        url = f"{BASE}/utils/{fname}"
        dst = os.path.join("utils", fname)
        if not os.path.exists(dst):
            urllib.request.urlretrieve(url, dst)
    if "utils" not in sys.path:
        sys.path.append(os.path.abspath("utils"))

def ensure_data():
    os.makedirs("data", exist_ok=True)
    for fname in ["sample_binary_losses.csv", "sample_real_losses.csv"]:
        url = f"{BASE}/data/{fname}"
        dst = os.path.join("data", fname)
        if not os.path.exists(dst):
            urllib.request.urlretrieve(url, dst)

ensure_utils()
ensure_data()
print("Bootstrap done: utils/ and data/ available.")


# E — aLTT: Adaptive selection with e-values (basic demo)

In [ ]:
import numpy as np, pandas as pd
from utils.csvio import load_losses_csv, is_binary_array

csv_path='data/sample_binary_losses.csv'
alpha_target=0.2
alpha_fwer=0.05
threshold = 1.0/alpha_fwer

ids, L, cols = load_losses_csv(csv_path)
rows=[]
for i, hp in enumerate(ids):
    losses=L[i]
    c=1.0
    E=1.0
    stop_t=None
    for t, ell in enumerate(losses, start=1):
        # bounded-loss surrogate e-increment (illustrative)
        inc = np.exp(c*(alpha_target - ell))
        E *= inc
        if E >= threshold and stop_t is None:
            stop_t = t
    rows.append({'hyperparam_id':hp,'E_final':E,'stop_round':stop_t})

pd.DataFrame(rows)